In [51]:
!pip install -q datasets

import re
import torch
import torch.nn as nn
import torch.optim as optim

from collections import Counter
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader

In [52]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

Device: cpu


In [54]:
from datasets import load_dataset

dataset = load_dataset("nyu-mll/glue", "sst2")

print(dataset)

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sst2/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 872
    })
    test: Dataset({
        features: ['sentence', 'label', 'idx'],
        num_rows: 1821
    })
})


In [55]:
train_data = dataset["train"].select(range(20000))
test_data = dataset["validation"]

print("Training samples:", len(train_data))
print("Testing samples:", len(test_data))

Training samples: 20000
Testing samples: 872


In [56]:
def tokenize(text):
    text = text.lower()
    return re.findall(r"\b\w+\b", text)

In [57]:
counter = Counter()

for item in train_data:
    counter.update(tokenize(item["sentence"]))

vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

max_vocab_size = 20000

for word, count in counter.most_common(max_vocab_size - 2):
    vocab[word] = len(vocab)

print("Vocabulary size:", len(vocab))

Vocabulary size: 12089


In [58]:
MAX_LEN = 64

def encode(text):
    tokens = tokenize(text)

    ids = [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

    ids = ids[:MAX_LEN]

    while len(ids) < MAX_LEN:
        ids.append(vocab["<PAD>"])

    return ids

In [59]:
class SentimentDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):

        text = self.data[index]["sentence"]
        label = self.data[index]["label"]

        input_ids = encode(text)

        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

In [60]:
train_dataset = SentimentDataset(train_data)
test_dataset = SentimentDataset(test_data)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

In [61]:
class BERTLikeSentimentModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_heads=4,
        num_layers=2,
        max_len=64,
        num_classes=2
    ):
        super().__init__()

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0
        )

        # Positional embeddings
        self.position_embedding = nn.Embedding(
            max_len,
            embedding_dim
        )

        # Transformer Encoder Layer
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=256,
            dropout=0.2,
            batch_first=True
        )

        # Multiple encoder layers
        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Dropout
        self.dropout = nn.Dropout(0.2)

        # Classification layer
        self.classifier = nn.Linear(
            embedding_dim,
            num_classes
        )

    def forward(self, input_ids):

        batch_size, seq_len = input_ids.shape

        # Position IDs
        positions = torch.arange(
            seq_len,
            device=input_ids.device
        ).unsqueeze(0).expand(
            batch_size,
            seq_len
        )

        # Token embeddings
        token_embeddings = self.token_embedding(
            input_ids
        )

        # Position embeddings
        position_embeddings = self.position_embedding(
            positions
        )

        # Combine token + position information
        x = token_embeddings + position_embeddings

        # Identify padding
        padding_mask = input_ids == 0

        # Transformer Encoder
        x = self.encoder(
            x,
            src_key_padding_mask=padding_mask
        )

        # Ignore padding during mean pooling
        mask = (~padding_mask).unsqueeze(-1)

        x = (x * mask).sum(dim=1)

        x = x / mask.sum(
            dim=1
        ).clamp(min=1)

        # Dropout
        x = self.dropout(x)

        # Classification
        output = self.classifier(x)

        return output

In [62]:
model = BERTLikeSentimentModel(
    vocab_size=len(vocab)
).to(device)

print(model)

BERTLikeSentimentModel(
  (token_embedding): Embedding(12089, 128, padding_idx=0)
  (position_embedding): Embedding(64, 128)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=256, bias=True)
        (dropout): Dropout(p=0.2, inplace=False)
        (linear2): Linear(in_features=256, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.2, inplace=False)
        (dropout2): Dropout(p=0.2, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [63]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0003,
    weight_decay=0.01
)

In [ ]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in train_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_accuracy = correct / total

    print(
        f"Epoch {epoch + 1}/{epochs} | "
        f"Loss: {total_loss / len(train_loader):.4f} | "
        f"Train Accuracy: {train_accuracy * 100:.2f}%"
    )

Epoch 1/5 | Loss: 0.6704 | Train Accuracy: 58.98%
Epoch 2/5 | Loss: 0.5945 | Train Accuracy: 68.08%


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in test_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

test_accuracy = correct / total

print(
    "Test Accuracy:",
    round(test_accuracy * 100, 2),
    "%"
)

In [ ]:
def predict_sentiment(text):

    model.eval()

    input_ids = torch.tensor(
        [encode(text)],
        dtype=torch.long
    ).to(device)

    with torch.no_grad():

        outputs = model(input_ids)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        prediction = torch.argmax(
            probabilities,
            dim=1
        ).item()

    if prediction == 1:
        label = "POSITIVE"
    else:
        label = "NEGATIVE"

    score = probabilities[0][prediction].item()

    return label, score

In [ ]:
test_sentences = [
    "I absolutely loved this movie!",
    "The movie was boring and disappointing.",
    "This is an excellent product.",
    "I hated the experience.",
    "The food was delicious and amazing.",
    "The service was terrible."
]

for text in test_sentences:

    label, score = predict_sentiment(text)

    print("Text:", text)
    print("Sentiment:", label)
    print("Score:", round(score * 100, 2), "%")
    print("-" * 60)